# Brain use 2 — the cerebral vessel network of a healthy brain

Body Atlas Models. **For learning anatomy, never for diagnosis.**

This kernel fetches the trainer from the public repository at a pinned commit, checks its
sha256, and runs it. Nothing is defined here: the notebook is transport, the script is the
record. That way the thing that ran and the thing in the repository are the same file, and
the result says which.

* labels: doi:10.5281/zenodo.17393202 — 100 healthy IXI brains, vessel masks, CC BY 4.0
* images: IXI MRA, CC BY-SA 3.0
* no patient data, no pre-training stage, five folds over the hundred


In [ ]:
!pip -q install nibabel
import subprocess, sys
print(subprocess.run([sys.executable, "-c",
    "import torch, scipy, skimage, nibabel;"
    "print('torch', torch.__version__, '| cuda', torch.cuda.is_available());"
    "print('scipy', scipy.__version__, '| skimage', skimage.__version__,"
    "'| nibabel', nibabel.__version__)"], capture_output=True, text=True).stdout)

In [ ]:
COMMIT = "PINNED_AT_PUSH_TIME"   # set to the commit that is being run, so the file is immutable
BASE = f"https://raw.githubusercontent.com/IAMOdon/body-atlas-models/{COMMIT}/organs/nervous-system/brain/healthy-vascular"

import hashlib, urllib.request, pathlib
for name in ("kaggle_train.py", "ixi_mra_index.json"):
    try:
        data = urllib.request.urlopen(f"{BASE}/{name}", timeout=120).read()
        pathlib.Path(name).write_bytes(data)
        print(f"{name}: {len(data):,d} bytes  sha256 {hashlib.sha256(data).hexdigest()}")
    except Exception as exc:
        print(f"{name}: not fetched ({type(exc).__name__}) "
              f"-- the trainer falls back to streaming the whole archive")

In [ ]:
!python kaggle_train.py --work-dir /kaggle/temp/ixi --out-dir /kaggle/working/out_use2

In [ ]:
# everything in one archive, ready to pull off the session
import pathlib
for p in sorted(pathlib.Path("/kaggle/working").glob("*")):
    print(f"{p.name:34s} {p.stat().st_size/1e6:9.2f} MB")